In [3]:
# ============================================================
# 03_keystroke_encoders.ipynb
# Adaptive Continuous Authentication — Keystroke Encoders
# ============================================================

"""
Goal:
  - Load keystroke_features.npz (20400 x 30)
  - Train:
      1) MLP classifier for user-ID (supervised)
      2) Autoencoder for unsupervised representation
  - Export:
      - keystroke_mlp.pt, keystroke_ae.pt
      - keystroke_embeddings_cls.npy, keystroke_embeddings_ae.npy
"""

# ============================================================
# Setup
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Paths
DATA_DIR = Path("data")
CKPT_DIR = Path("checkpoints"); CKPT_DIR.mkdir(exist_ok=True)
EMB_DIR = Path("embeddings");   EMB_DIR.mkdir(exist_ok=True)

FEATURES_PATH = DATA_DIR / "keystroke_features.npz"

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ============================================================
# 1. Load features
# ============================================================

z = np.load(FEATURES_PATH, allow_pickle=True)
X = z["features"].astype(np.float32)   # (N, 30)
user_ids = z["user_id"].astype(str)   # (N,)
session_ids = z["session_id"].astype(str)

print("Feature matrix:", X.shape)
print("Unique users:", len(np.unique(user_ids)))

# Label-encode user IDs
le_users = LabelEncoder()
y = le_users.fit_transform(user_ids)   # 0..num_users-1
num_users = len(le_users.classes_)
print("Encoded users:", num_users)

# Standardize features (helps both MLP & AE)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)

# Train/val split
X_tr, X_va, y_tr, y_va = train_test_split(
    X_scaled, y, test_size=0.2, random_state=SEED, stratify=y
)
print("Train:", X_tr.shape, "Val:", X_va.shape)

# ============================================================
# 2. Dataset / Dataloaders
# ============================================================

class KeystrokeDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.from_numpy(X).float()
        self.y = None if y is None else torch.from_numpy(y).long()
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        if self.y is None:
            return self.X[idx]
        return self.X[idx], self.y[idx]

batch_size_cls = 64
batch_size_ae  = 128

ds_tr = KeystrokeDataset(X_tr, y_tr)
ds_va = KeystrokeDataset(X_va, y_va)
dl_tr = DataLoader(ds_tr, batch_size=batch_size_cls, shuffle=True)
dl_va = DataLoader(ds_va, batch_size=batch_size_cls, shuffle=False)

# ============================================================
# 3. MLP Classifier Encoder
# ============================================================

class MLPEncoder(nn.Module):
    """
    Simple classifier:
      in_dim -> 128 -> 64 (embedding) -> num_users
    """
    def __init__(self, in_dim, num_classes, dropout=0.1):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
        )
        self.head = nn.Linear(64, num_classes)

    def forward(self, x, return_embed=False):
        z = self.backbone(x)
        logits = self.head(z)
        if return_embed:
            return logits, z
        return logits

def train_classifier(X_tr, y_tr, X_va, y_va, in_dim, num_classes,
                     epochs=40, lr=1e-3, batch_size=64):
    ds_tr = KeystrokeDataset(X_tr, y_tr)
    ds_va = KeystrokeDataset(X_va, y_va)
    dl_tr = DataLoader(ds_tr, batch_size=batch_size, shuffle=True)
    dl_va = DataLoader(ds_va, batch_size=batch_size, shuffle=False)

    model = MLPEncoder(in_dim, num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr)

    best_state = None
    best_acc = 0.0

    for epoch in range(1, epochs + 1):
        # Train
        model.train()
        for xb, yb in dl_tr:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        # Validate
        model.eval()
        preds, gts = [], []
        with torch.no_grad():
            for xb, yb in dl_va:
                xb = xb.to(device)
                logits = model(xb)
                pred = logits.argmax(dim=1).cpu().numpy()
                preds.append(pred)
                gts.append(yb.numpy())
        preds = np.concatenate(preds)
        gts = np.concatenate(gts)
        acc = accuracy_score(gts, preds)

        if acc > best_acc:
            best_acc = acc
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}

        if epoch == 1 or epoch % 5 == 0:
            print(f"[CLS] epoch {epoch:02d} val_acc={acc:.4f}")

    if best_state is None:
        print("⚠️ No improvement tracked, using last model state.")
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    print(f"✅ Best classifier val_acc={best_acc:.4f}")
    return model, best_acc

print("\n🔧 Training MLP classifier encoder...")
cls_model, cls_acc = train_classifier(
    X_tr, y_tr, X_va, y_va,
    in_dim=X_scaled.shape[1],
    num_classes=num_users,
    epochs=40, lr=1e-3, batch_size=batch_size_cls
)

# Save classifier checkpoint
cls_ckpt_path = CKPT_DIR / "keystroke_mlp.pt"
torch.save({k: v.cpu() for k, v in cls_model.state_dict().items()}, cls_ckpt_path)
print("💾 Saved classifier checkpoint →", cls_ckpt_path)

# ============================================================
# 4. Autoencoder
# ============================================================

class KeystrokeAE(nn.Module):
    """
    Simple symmetric autoencoder:
      in_dim -> 64 -> 16 (bottleneck) -> 64 -> in_dim
    """
    def __init__(self, in_dim):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(in_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 16),
            nn.ReLU()
        )
        self.dec = nn.Sequential(
            nn.Linear(16, 64),
            nn.ReLU(),
            nn.Linear(64, in_dim)
        )

    def forward(self, x, return_embed=False):
        z = self.enc(x)
        x_hat = self.dec(z)
        if return_embed:
            return x_hat, z
        return x_hat

def train_autoencoder(X_all, in_dim, epochs=60, lr=1e-3, batch_size=128):
    ds = KeystrokeDataset(X_all)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True)

    model = KeystrokeAE(in_dim).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr)

    best_state = None
    best_loss = float("inf")

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        n = 0
        for xb in dl:
            xb = xb.to(device)
            optimizer.zero_grad()
            x_hat = model(xb)
            loss = criterion(x_hat, xb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * xb.size(0)
            n += xb.size(0)
        epoch_loss = total_loss / max(n, 1)

        if epoch == 1 or epoch % 5 == 0:
            print(f"[AE ] epoch {epoch:02d} recon_loss={epoch_loss:.6f}")

        if epoch_loss < best_loss:
            best_loss = epoch_loss
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}

    if best_state is None:
        print("⚠️ No improvement tracked, using last AE state.")
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    print(f"✅ Best AE recon_loss={best_loss:.6f}")
    return model, best_loss

print("\n🔧 Training autoencoder...")
ae_model, ae_loss = train_autoencoder(
    X_scaled, in_dim=X_scaled.shape[1],
    epochs=60, lr=1e-3, batch_size=batch_size_ae
)

# Save AE checkpoint
ae_ckpt_path = CKPT_DIR / "keystroke_ae.pt"
torch.save({k: v.cpu() for k, v in ae_model.state_dict().items()}, ae_ckpt_path)
print("💾 Saved AE checkpoint →", ae_ckpt_path)

# ============================================================
# 5. Export embeddings for all samples
# ============================================================

print("\n🔎 Exporting embeddings for all samples...")

# Classifier embeddings (64-d)
cls_model.eval()
with torch.no_grad():
    X_torch = torch.from_numpy(X_scaled).float().to(device)
    _, Z_cls = cls_model(X_torch, return_embed=True)
    Z_cls = Z_cls.cpu().numpy().astype(np.float32)

# AE bottleneck embeddings (16-d)
ae_model.eval()
with torch.no_grad():
    X_torch = torch.from_numpy(X_scaled).float().to(device)
    _, Z_ae = ae_model(X_torch, return_embed=True)
    Z_ae = Z_ae.cpu().numpy().astype(np.float32)

emb_cls_path = EMB_DIR / "keystroke_embeddings_cls.npy"
emb_ae_path  = EMB_DIR / "keystroke_embeddings_ae.npy"

np.save(emb_cls_path, Z_cls)
np.save(emb_ae_path, Z_ae)

print("💾 Saved classifier embeddings →", emb_cls_path, "shape:", Z_cls.shape)
print("💾 Saved AE embeddings →", emb_ae_path, "shape:", Z_ae.shape)

# ============================================================
# 6. Summary
# ============================================================

print("\n===== Summary =====")
print(f"MLP classifier val accuracy: {cls_acc:.4f}")
print(f"AE best recon loss         : {ae_loss:.6f}")
print("Embeddings CLS shape       :", Z_cls.shape)
print("Embeddings AE shape        :", Z_ae.shape)
print("Checkpoints saved in       :", CKPT_DIR.resolve())
print("Embeddings saved in        :", EMB_DIR.resolve())

Using device: cpu
Feature matrix: (20400, 30)
Unique users: 51
Encoded users: 51
Train: (16320, 30) Val: (4080, 30)

🔧 Training MLP classifier encoder...
[CLS] epoch 01 val_acc=0.4600
[CLS] epoch 05 val_acc=0.5569
[CLS] epoch 10 val_acc=0.6010
[CLS] epoch 15 val_acc=0.6142
[CLS] epoch 20 val_acc=0.6272
[CLS] epoch 25 val_acc=0.6299
[CLS] epoch 30 val_acc=0.6407
[CLS] epoch 35 val_acc=0.6370
[CLS] epoch 40 val_acc=0.6544
✅ Best classifier val_acc=0.6544
💾 Saved classifier checkpoint → checkpoints/keystroke_mlp.pt

🔧 Training autoencoder...
[AE ] epoch 01 recon_loss=0.391631
[AE ] epoch 05 recon_loss=0.028019
[AE ] epoch 10 recon_loss=0.018855
[AE ] epoch 15 recon_loss=0.012218
[AE ] epoch 20 recon_loss=0.009072
[AE ] epoch 25 recon_loss=0.008162
[AE ] epoch 30 recon_loss=0.006306
[AE ] epoch 35 recon_loss=0.004996
[AE ] epoch 40 recon_loss=0.004771
[AE ] epoch 45 recon_loss=0.004338
[AE ] epoch 50 recon_loss=0.004482
[AE ] epoch 55 recon_loss=0.003820
[AE ] epoch 60 recon_loss=0.004079
